# 06 · Feature Engineering

## Objective
Test whether a small number of **justified** new features improve cross-validated performance.
Features are kept only if the evidence supports them.

Everything here uses the **training set only**; the test set is not touched.

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can use src/churn_utils.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import inspect

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate

from src.churn_utils import RANDOM_STATE, add_engineered_features, build_pipeline, load_split

X_train, X_test, y_train, y_test = load_split()
print("Train:", X_train.shape)

Train: (5634, 19)


## 1. Candidate features (motivated by EDA)

| Feature | Idea | Why it might help |
|---|---|---|
| `num_addon_services` | Count of the 6 add-ons (security, backup, protection, support, TV, movies) the customer has | Captures overall **engagement / switching cost** in one number instead of six separate columns |
| `tenure_group` | Tenure bands: 0–12, 13–24, 25–48, 49+ months | EDA showed churn drops **non-linearly** with tenure; a linear model can't bend the curve from `tenure` alone |

**Considered and rejected:** `TotalCharges / tenure` (average historical charge). It is almost identical to
`MonthlyCharges`, so it would be redundant.

Both features are computed **row by row** (no statistics learned from the data), so they cannot leak information.
They are added inside the pipeline with a `FunctionTransformer`:

In [2]:
print(inspect.getsource(add_engineered_features))

def add_engineered_features(X):
    """Row-wise features (no statistics learned from data -> no leakage risk)."""
    X = X.copy()
    # How many optional add-on services the customer subscribes to (0-6)
    X["num_addon_services"] = (X[ADDON_SERVICES] == "Yes").sum(axis=1)
    # Customer lifecycle stage; churn risk is highly non-linear in tenure
    X["tenure_group"] = pd.cut(
        X["tenure"], bins=[-1, 12, 24, 48, np.inf],
        labels=["0-12m", "13-24m", "25-48m", "49m+"],
    ).astype(str)
    return X



In [3]:
train_fe = add_engineered_features(X_train).assign(Churn=y_train.values)

addon_summary = train_fe.groupby("num_addon_services")["Churn"].agg(churn_rate="mean", customers="size")
tenure_summary = train_fe.groupby("tenure_group")["Churn"].agg(churn_rate="mean", customers="size")
display(addon_summary.round(3))
display(tenure_summary.round(3))

,churn_rate,customers
num_addon_services,,
0,0.209,1757
1,0.458,766
2,0.362,832
3,0.278,885
4,0.230,695
5,0.125,464
6,0.047,235


,churn_rate,customers
tenure_group,,
0-12m,0.473,1737
13-24m,0.286,818
25-48m,0.202,1290
49m+,0.099,1789


**Observations (training data)**
- `tenure_group` shows a clear monotonic decline: 47.3% churn in the first year → 9.9% after 4 years.
- `num_addon_services` is **not monotonic**: customers with 0 add-ons churn *less* (20.9%) than those with 1 (45.8%),
  because the "0" group includes all customers without internet (who rarely churn). From 1 add-on upward,
  churn falls steadily (45.8% → 4.7%). The original columns (`InternetService` + the six add-ons) already
  carry this information.

## 2. Does it help? Cross-validated comparison

Compare the **same model** with and without the engineered features using 5-fold stratified CV on the
training set. Two model families are used because a feature may help a linear model but not a tree model
(trees can already learn non-linear tenure effects).

Metrics: **ROC-AUC** (ranking quality, threshold-independent) and **PR-AUC / average precision**
(focuses on the churn class).

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                            random_state=RANDOM_STATE, n_jobs=-1),
}

rows = []
for model_name, model in models.items():
    for engineered in [False, True]:
        scores = cross_validate(build_pipeline(model, engineered=engineered), X_train, y_train,
                                cv=cv, scoring=["roc_auc", "average_precision"])
        rows.append({
            "model": model_name,
            "features": "original + engineered" if engineered else "original",
            "roc_auc_mean": scores["test_roc_auc"].mean(),
            "roc_auc_std": scores["test_roc_auc"].std(),
            "pr_auc_mean": scores["test_average_precision"].mean(),
        })

comparison = pd.DataFrame(rows)
comparison.round(4)

,model,features,roc_auc_mean,roc_auc_std,pr_auc_mean
0,Logistic Regression,original,0.8462,0.0126,0.6615
1,Logistic Regression,original + engineered,0.8465,0.0116,0.6651
2,Random Forest,original,0.8446,0.0095,0.6571
3,Random Forest,original + engineered,0.8451,0.0097,0.6586


**Observations**
- Logistic Regression: ROC-AUC 0.8462 → 0.8465 (+0.0003); PR-AUC 0.6615 → 0.6651.
- Random Forest: ROC-AUC 0.8446 → 0.8451 (+0.0005); PR-AUC 0.6571 → 0.6586.
- The improvements are **far smaller than the fold-to-fold standard deviation** (~0.01), i.e. they are
  within noise and not meaningful.

## Conclusion

**Decision: do NOT use the engineered features in the final model.**

**Reason:** they add complexity without a measurable improvement. The original features already describe
the same information: one-hot encoded add-on columns cover service engagement, and the models can use
`tenure` together with `Contract` to capture the early-tenure risk.

Notebook 07 therefore uses `build_pipeline(model, engineered=False)`. The feature code stays in
`src/churn_utils.py` so this experiment remains reproducible.

> Lesson: feature engineering is an experiment, not a mandatory step. A negative result is a valid
> result and is worth reporting.